# Performing sweeps

In this example notebook, we use the gnm package to perform a sweep over generative rules and parameter values.

In [1]:
import os
notebook_dir = os.getcwd() 

# able to run example script from within Jupyter notebook in package
import sys
loc = os.path.abspath(os.path.join(notebook_dir, '..', '..', 'src'))
sys.path.append(loc)

# other basic imports
import time
import torch
from gnm import *
from gnm import defaults, utils, evaluation, fitting, generative_rules, weight_criteria

In [2]:
# device is the GPU if available, otherwise the CPU - GPU is much faster but ensure you have 
# GPU available in your PC/HPC

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## Loading in data

We'll start by loading in some data for our sweep. In particular, we'll need:
1. A distance matrix
2. A binary network to compare our networks to

We'll get these from the defaults sub-module, which has a built in distance matrix and consensus network.

In [3]:
distance_matrix = defaults.get_distance_matrix(device=DEVICE)
binary_consensus_network = defaults.get_binary_network(device=DEVICE)

binary_consensus_network = torch.stack([binary_consensus_network, binary_consensus_network], dim=0)
binary_consensus_network = binary_consensus_network.squeeze(1)

print(f"Distance matrix shape: {distance_matrix.shape}")
print(f"Binary consensus network shape: {binary_consensus_network.shape}")

Distance matrix shape: torch.Size([90, 90])
Binary consensus network shape: torch.Size([2, 90, 90])


We'll run the models until they have the same number of connections as the real binary consensus network

In [4]:
num_connections = int( binary_consensus_network.sum().item() / 2 / binary_consensus_network.shape[0] )
print(f"The binary consensus network contains {num_connections} connections.")

The binary consensus network contains 400 connections.


## Defining our sweep

The next step is to define the parameters we want to sweep over. Here, we'll sweep over a range of values for $\eta$ and $\gamma$, while keeping the generative rule fixed (using the Matching Index) and the weight optimisation criterion fixed (using the distance weighted communicability).   

For each set of parameters, we'll generate 100 networks using the model with that set of parameters. This means setting the number of simulations to 100. 

In [ ]:
eta_values_sparse = torch.linspace(-10, 10, 5)
gamma_values_sparse = torch.linspace(-10, 10, 5)

generative_rules = [generative_rules.MatchingIndex()]

binary_sweep_parameters_sparse = fitting.BinarySweepParameters(
    eta = eta_values_sparse,
    gamma = gamma_values_sparse,
    lambdah = torch.Tensor([0.0]),
    distance_relationship_type = ["powerlaw"],
    preferential_relationship_type = ["powerlaw"],
    heterochronicity_relationship_type = ["powerlaw"],
    generative_rule = generative_rules,
    num_iterations = [num_connections],
)

num_simulations = 10

sweep_config_sparse = fitting.SweepConfig(
    binary_sweep_parameters = binary_sweep_parameters_sparse,
    num_simulations = num_simulations,
    distance_matrix = [distance_matrix]    
)

## Creating our evaluations

We want to evaluate how good the fit of our models is the real binary consensus network.
For our evaluation criteria, we'll use the maximum of the KS statistics across clustering coefficient, degree, and edge length distributions.  

In [6]:
criteria = [ evaluation.ClusteringKS(), evaluation.DegreeKS(), evaluation.EdgeLengthKS(distance_matrix) ]
energy = evaluation.MaxCriteria( criteria )
binary_evaluations = [energy]

## Performing the sweep

In [7]:

start_time = time.perf_counter()

experiments_sparse = fitting.perform_sweep(sweep_config=sweep_config_sparse, 
                                binary_evaluations=binary_evaluations, 
                                real_binary_matrices=binary_consensus_network,
                                save_model = False,
                                save_run_history = False,
                                verbose = True
)

end_time = time.perf_counter()

Using device: cpu for GNM simulations


Configuration Iterations:   0%|          | 0/9 [00:00<?, ?it/s]/Users/will/Documents/GenerativeNetworkModels/src/gnm/model.py:784: UserWarning: Numerical overflow detected in wiring probabilities (eta=-10.0, gamma=-10.0, lambdah=0.0, relationship_types=powerlaw/powerlaw/powerlaw). Values clamped to safe range. Further occurrences will be counted silently.
  warnings.warn(
Configuration Iterations:  33%|███▎      | 3/9 [00:03<00:07,  1.29s/it]/Users/will/Documents/GenerativeNetworkModels/src/gnm/model.py:784: UserWarning: Numerical overflow detected in wiring probabilities (eta=0.0, gamma=-10.0, lambdah=0.0, relationship_types=powerlaw/powerlaw/powerlaw). Values clamped to safe range. Further occurrences will be counted silently.
  warnings.warn(
Configuration Iterations:  67%|██████▋   | 6/9 [00:07<00:03,  1.26s/it]/Users/will/Documents/GenerativeNetworkModels/src/gnm/model.py:784: UserWarning: Numerical overflow detected in wiring probabilities (eta=10.0, gamma=-10.0, lambdah=0.0, rel


Average time per run: 1.22 seconds
Total time for sweep: 10.99 seconds


Let's look at the efficiency of the sweep.

In [8]:
print(f"Sweep took {end_time - start_time:0.3f} seconds.")

total_simulations = num_simulations * len(eta_values_sparse) * len(gamma_values_sparse)

print(f"Total number of simulations: {total_simulations}")

print(f"Average time per simulation: {(end_time - start_time) / total_simulations:0.3f} seconds.")

Sweep took 11.251 seconds.
Total number of simulations: 90
Average time per simulation: 0.125 seconds.


## Analysing the results

In [9]:

# Get all optimal experiments and their KS energies based on the energy criterion
# The lower the energy, the better the fit to the real data. The optimal experiments are those with the lowest energy values.
optimal_experiments, optimal_energies = fitting.optimise_evaluation(
    experiments=experiments_sparse,
    criterion=energy,
)

# get optimal eta and gamma values to set the range for the coarse sweep.
offset = 1.5

optimal_etas = [experiment.run_config.binary_parameters.eta for experiment in optimal_experiments]
optimal_eta_min, optimal_eta_max = min(optimal_etas) - offset, max(optimal_etas) + offset

optimal_gammas = [experiment.run_config.binary_parameters.gamma for experiment in optimal_experiments]
optimal_gamma_min, optimal_gamma_max = min(optimal_gammas) - offset, max(optimal_gammas) + offset

print(f"Optimal eta range: {optimal_eta_min} to {optimal_eta_max}")
print(f"Optimal gamma range: {optimal_gamma_min} to {optimal_gamma_max}")

Optimal eta range: -1.5 to 1.5
Optimal gamma range: 8.5 to 11.5


In [10]:
from gnm.fitting.experiment_saving import ExperimentEvaluation


eval_handler = ExperimentEvaluation('./experiment_results')


# Save the list of experiments 
eval_handler.save_experiments(experiments_sparse)
gnm_results_dataframe = eval_handler.get_dataframe_of_results([
'eta',
'gamma', 'mean_of_max_ks_per_connectome', 'generative_rule'
])
eval_handler.list_experiment_parameters()

Total entries for connectome_index: 23460
Total entries for eta: 23460
Total entries for gamma: 23460
Total entries for mean_of_max_ks_per_connectome: 23460
Total entries for generative_rule: 23460
Compiled results for 10 unique connectomes.
Experiment Parameters:
   - eta
   - gamma
   - lambdah
   - distance_relationship_type
   - preferential_relationship_type
   - heterochronicity_relationship_type
   - generative_rule
   - num_iterations
   - prob_offset
   - binary_updates_per_iteration
   - n_participants
   - mean_of_max_ks_per_connectome
   - std_of_max_ks_per_connectome
   - per_connectome_binary_evals


/Users/will/Documents/GenerativeNetworkModels/src/gnm/fitting/experiment_saving.py:692: UserWarning: Assumes default Max KS Distance metric was used during evaluation.
  warn("Assumes default Max KS Distance metric was used during evaluation.")


dict_keys(['eta', 'gamma', 'lambdah', 'distance_relationship_type', 'preferential_relationship_type', 'heterochronicity_relationship_type', 'generative_rule', 'num_iterations', 'prob_offset', 'binary_updates_per_iteration', 'n_participants', 'mean_of_max_ks_per_connectome', 'std_of_max_ks_per_connectome', 'per_connectome_binary_evals'])

In [ ]:
# repeat for coarse sweep with more points in the optimal range of eta and gamma for higher resolution of the optimal parameter space.

binary_sweep_parameters_coarse = binary_sweep_parameters_sparse
binary_sweep_parameters_coarse.eta = torch.linspace(optimal_eta_min, optimal_eta_max, 30)
binary_sweep_parameters_coarse.gamma = torch.linspace(optimal_gamma_min, optimal_gamma_max, 30)

sweep_config_coarse = fitting.SweepConfig(
    binary_sweep_parameters = binary_sweep_parameters_coarse,
    num_simulations = num_simulations,
    distance_matrix = [distance_matrix]
)

experiments_coarse = fitting.perform_sweep(sweep_config=sweep_config_coarse,
                                binary_evaluations=binary_evaluations,
                                real_binary_matrices=binary_consensus_network,
                                save_model = False,
                                save_run_history = False,
)


optimal_experiments, optimal_energies = fitting.optimise_evaluation(
    experiments=experiments_sparse,
    criterion=energy,
)

optimal_etas = [experiment.run_config.binary_parameters.eta for experiment in optimal_experiments]
optimal_gammas = [experiment.run_config.binary_parameters.gamma for experiment in optimal_experiments]

print(f'Optimal Etas: {optimal_etas}')
print(f'Optimal Gammas: {optimal_gammas}')

Using device: cpu for GNM simulations
Optimal Etas: [tensor(0.), tensor(0.)]
Optimal Gammas: [tensor(10.), tensor(10.)]
